In [1]:
!git clone https://github.com/OpenMOSS/MOSS-TTS.git
%cd MOSS-TTS
!pip install datasets soundfile -q wandb
!apt-get install -y ffmpeg -q

fatal: destination path 'MOSS-TTS' already exists and is not an empty directory.
/content/MOSS-TTS
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
# MODEL_NAME = "OpenMOSS-Team/MOSS-TTS"                 # Delay 8B
MODEL_NAME = "OpenMOSS-Team/MOSS-TTS-Local-Transformer" # Local 1.7B

N_SAMPLES = 20
MIN_WORDS = 10
MAX_WORDS = 60
MAX_NEW_TOKENS = 2000
OUTPUT_DIR = "/content/profiling_results_hf"

In [2]:
import json, random, time
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
from datasets import load_dataset
from transformers import AutoModel, AutoProcessor

import wandb

# required SDPA backend flags from official MOSS-TTS docs
torch.backends.cuda.enable_cudnn_sdp(False)
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{device}')

cuda


In [3]:
#sample runner: run through samples and collect correct metrics
def run_one_sample(model, processor, text, output_dir, idx):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    batch = processor([[processor.build_user_message(text=text)]], mode='generation')
    inputs = {k: v.to(device) for k, v in batch.items()}

    #synchronize and reset GPU mem stats before generation
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    t0 = time.perf_counter()
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    

    torch.cuda.synchronize()

    # get memory and timing stats
    totalS = time.perf_counter() - t0
    peakMB = torch.cuda.max_memory_allocated() / 1e6

    # decode audio and save
    messages  = processor.decode(out)
    audio = messages[0].audio_codes_list[0].cpu().float()
    sr = processor.model_config.sampling_rate
    audioDur = audio.shape[-1] / sr

    sf.write(str(Path(output_dir) / f'sample_{idx:03d}.wav'), audio.squeeze().numpy(), sr)

    return {'text': text, 'word_count': len(text.split()),
            'total_time_s': round(totalS, 3),
            'audio_dur_s': round(audioDur, 3),
            'rtf': round(totalS / audioDur, 4),
            'peak_gpu_mb': round(peakMB, 1)}

In [4]:
# load dataset
ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='test')

texts = [row['text'].strip() for row in ds if MIN_WORDS < len(row['text'].split()) <= MAX_WORDS]
random.seed(42)

texts = random.sample(texts, min(N_SAMPLES, len(texts)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
# model and processor loading
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)

if hasattr(processor, 'audio_tokenizer'):
    processor.audio_tokenizer = processor.audio_tokenizer.to(device).eval()

model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.bfloat16).to(device).eval()

Loading weights:   0%|          | 0/1600 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/463 [00:00<?, ?it/s]

In [6]:
# warmup 
warmup = processor([[processor.build_user_message(text='Hello warmup.')]], mode='generation')

with torch.no_grad():
    _ = model.generate(**{k: v.to(device) for k, v in warmup.items()}, max_new_tokens=200)

Generating bs1 ...:  24%|██▍       | 48/200 [00:03<00:10, 14.72it/s]


In [7]:
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)


wandbRun = wandb.init(project="hpml-final-project", name= f"hf-{MODEL_NAME}-inference-profile")

# for texts run sample and collect res
for i, text in enumerate(texts, 1):

    run = run_one_sample(model, processor, text, str(out/'wav'), i)

    print(f'[{i:2d}/{len(texts)}] {run["word_count"]:3d}w  'f'RTF={run["rtf"]:.3f}  dur={run["audio_dur_s"]:.1f}s total={run["total_time_s"]:.1f}s  peak={run["peak_gpu_mb"]:.0f}MB')
    wandbRun.log({"text": text, "word_count": run["word_count"], "rtf": run["rtf"], "audio_dur_s": run["audio_dur_s"], "peak_gpu_mb": run["peak_gpu_mb"], "total_time_s": run["total_time_s"]})

wandbRun.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ac5905 (ac5905-columbia-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Generating bs1 ...:  10%|▉         | 198/2000 [00:11<01:46, 16.96it/s]


[ 1/20]  50w  RTF=0.885  dur=13.2s total=11.7s  peak=24325MB


Generating bs1 ...:   5%|▍         | 98/2000 [00:05<01:52, 16.97it/s]


[ 2/20]  13w  RTF=1.112  dur=5.2s total=5.8s  peak=24286MB


...

Generating bs1 ...:   3%|▎         | 64/2000 [00:03<01:56, 16.56it/s]


[19/20]  13w  RTF=1.561  dur=2.5s total=3.9s  peak=24286MB


Generating bs1 ...:   4%|▍         | 85/2000 [00:04<01:52, 17.01it/s]


[20/20]  15w  RTF=1.202  dur=4.2s total=5.0s  peak=24294MB


audio_dur_s,▅▂▂▁▁▃▆▅▇▂█▄▅▄▇▃▁▃▁▂
peak_gpu_mb,▆▁▁▁▁▂▆▅▅▂█▄▄▄▆▃▁▂▁▂
rtf,▁▃▄▇█▂▁▁▁▃▁▂▂▂▁▂█▂▇▄
total_time_s,▅▂▂▁▁▃▆▅▇▂█▄▅▄▇▃▁▃▁▂
word_count,▇▁▁▁▁▂▆▇█▁█▆▆▄▇▃▁▂▁▂
audio_dur_s,4.16
peak_gpu_mb,24293.9
rtf,1.2023
text,= = = = 2015 season ...
total_time_s,5.002
word_count,15
